# 00_env_setup - Azure OpenAI Responses + Agent Framework

Quick checks for environment setup. This notebook verifies required environment variables and performs a short Azure OpenAI Responses ping via `agent-framework-azure-ai`. Agent Service connectivity stays stubbed until the hosted SDK/version is pinned.

## Prerequisites
- Python 3.11+ with this repo's virtual environment activated.
- Dependencies installed: see `requirements.txt` (run the `%pip` cell below if needed).
- Environment variables set:
  - `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_API_VERSION` (defaults to `2024-02-15-preview`).
  - `AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME` (Responses deployment for `AzureOpenAIResponsesClient`).
  - Optional: `AZURE_OPENAI_CHAT_DEPLOYMENT_NAME`, `AZURE_OPENAI_BASE_URL`.
  - `AZURE_AGENT_SERVICE_ENDPOINT`, `AZURE_AGENT_SERVICE_API_KEY` (or AAD) for Agent Service once the SDK is added.

In [ ]:
# Run once per environment to install dependencies if needed
# %pip install -U pip -r ../requirements.txt

In [ ]:
import os
from typing import List

required_openai_vars: List[str] = [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_API_VERSION",
    "AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME",
]
optional_openai_vars: List[str] = [
    "AZURE_OPENAI_CHAT_DEPLOYMENT_NAME",
    "AZURE_OPENAI_BASE_URL",
]
agent_service_vars: List[str] = [
    "AZURE_AGENT_SERVICE_ENDPOINT",
    "AZURE_AGENT_SERVICE_API_KEY",
]


def check_env(names: List[str], label: str) -> List[str]:
    print(f"\n{label}")
    missing: List[str] = []
    for name in names:
        if os.environ.get(name):
            print(f"- {name}: set")
        else:
            print(f"- {name}: MISSING")
            missing.append(name)
    return missing


missing_core = check_env(required_openai_vars, "Azure OpenAI (required for Responses)")
check_env(optional_openai_vars, "Azure OpenAI (optional)")
missing_agent = check_env(agent_service_vars, "Agent Service (optional until SDK pinned)")

if missing_core:
    print("\nSet the missing Azure OpenAI variables before running the ping below.")
else:
    print("\nAzure OpenAI variables are present.")
if missing_agent:
    print("Agent Service variables not fully set; skip Agent Service calls until configured.")

In [ ]:
import asyncio
from agent_framework.azure import AzureOpenAIResponsesClient
from azure.identity import AzureCliCredential

RUN_AZURE_AGENT_PING = True  # flip to False to skip the call during quick iterations
USE_CLI_CREDENTIAL = False  # set True to force Azure CLI auth instead of API key

if RUN_AZURE_AGENT_PING:
    endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT")
    deployment = os.environ.get("AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME")
    api_version = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-02-15-preview")
    api_key = os.environ.get("AZURE_OPENAI_API_KEY")
    base_url = os.environ.get("AZURE_OPENAI_BASE_URL")

    if not endpoint or not deployment:
        raise RuntimeError(
            "Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME to run the ping."
        )

    credential = None
    if USE_CLI_CREDENTIAL or not api_key:
        credential = AzureCliCredential()
        if not api_key:
            print("Using AzureCliCredential because AZURE_OPENAI_API_KEY is missing.")

    client = AzureOpenAIResponsesClient(
        endpoint=endpoint,
        deployment_name=deployment,
        api_version=api_version,
        api_key=api_key if api_key else None,
        credential=credential,
        base_url=base_url,
    )

    async def ping() -> None:
        agent = client.create_agent(
            name="EnvCheckAgent",
            instructions="You return a short healthcheck message.",
        )
        reply = await agent.run("Reply with 'pong' to confirm connectivity.")
        print("Agent reply:", reply)

    await ping()
else:
    print("Azure OpenAI ping skipped (RUN_AZURE_AGENT_PING=False).")

In [ ]:
# Agent Service placeholder
print(
    "Agent Service connectivity test goes here once the hosted SDK/version is chosen.\n"
    "Expected flow: instantiate the client, list or deploy an agent, then send a simple message."
)

## Next steps
- Confirm the Agent Service SDK/version and add the import + ping cell above.
- Move on to `01_responses_api_basics.ipynb` once the Azure OpenAI ping succeeds.